In [ ]:
# Import core libraries: numpy, pandas, matplotlib, seaborn.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Configure pandas to display all columns/rows without truncation.
pd.set_option('display.max_columns', None)   
pd.set_option('display.max_rows', None)      
pd.set_option('display.width', None)    

## Reading Data

In [ ]:
# Load the solar farm CSV into a DataFrame.
df = pd.read_csv("Synthetic-Solar-Farm-Stream-No-Repair.csv")

In [ ]:
# Check the shape (rows, columns) of the data.
df.shape

In [ ]:
# Preview the first 5 rows.
df.head()

In [ ]:
# Preview the last 5 rows.
df.tail()

In [ ]:
# List all column names.
df.columns.tolist()

In [ ]:
# Show dtypes and non-null counts per column.
df.info()

## Data Quality

In [ ]:
# Count missing values per column.
df.isnull().sum()

In [ ]:
# Table of missing-value counts and percentages, sorted descending.
missing = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

missing.sort_values("Missing Count", ascending=False)

In [ ]:
# Count fully duplicated rows.
df.duplicated().sum()

In [ ]:
# Heatmap of missing values across columns and rows.
plt.figure(figsize=(10,5))
plt.title("Missing Values Heatmap")
sns.heatmap(df.isnull(),yticklabels=False)
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.show()

## Target Analysis

In [ ]:
# Value counts for is_faulted.
df["is_faulted"].value_counts()

In [ ]:
# Value counts for fault_severity.
df['fault_severity'].value_counts()

In [ ]:
# Percentage breakdown of is_faulted.
df["is_faulted"].value_counts(normalize=True) * 100

In [ ]:
# Countplot of healthy vs faulted observations.
plt.figure(figsize=(7, 5))
plt.title("Target Distribution")
sns.countplot(data=df,x="is_faulted")
plt.xlabel("Fault Status")
plt.ylabel("Number of Observations")

plt.show()

## Device Profile

In [ ]:
# Average fault rate (is_faulted) per device.
device_fault_ratio = df.groupby("device")["is_faulted"].mean() * 100
device_fault_ratio

In [ ]:
# Bar chart of fault ratio per device.
plt.figure(figsize=(10, 5))
plt.title("Fault Ratio per Device")
sns.barplot(x=device_fault_ratio.index, y=device_fault_ratio.values)
plt.xlabel("Device")
plt.ylabel("Fault Ratio (%)")
plt.show()

In [ ]:
# Crosstab of fault type counts per device.
device_fault_types = pd.crosstab(df["device"], df["fault_labels"])
device_fault_types

In [ ]:
# Stacked bar chart of fault types per device.
device_fault_types.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)
plt.title("Fault Type Distribution per Device")
plt.xlabel("Device")
plt.ylabel("Count")
plt.legend(title="fault_labels", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# Average active_power per device.
device_power = df.groupby("device")["active_power"].mean()
device_power

In [ ]:
# Bar chart of average active power per device.
plt.figure(figsize=(10, 5))
plt.title("Average Active Power per Device")
sns.barplot(x=device_power.index, y=device_power.values)
plt.xlabel("Device")
plt.ylabel("Average Active Power")
plt.show()

In [ ]:
# Average performance_ratio per device.
device_pr = df.groupby("device")["performance_ratio"].mean()
device_pr

In [ ]:
# Bar chart of average performance ratio per device.
plt.figure(figsize=(10, 5))
plt.title("Average Performance Ratio per Device")
sns.barplot(x=device_pr.index, y=device_pr.values)
plt.xlabel("Device")
plt.ylabel("Average Performance Ratio")
plt.show()

## Numerical Features Analysis

In [ ]:
# List numeric-only columns.
df.select_dtypes(include=np.number).columns

In [ ]:
# Numeric feature columns, excluding the target is_faulted.
numeric_features = df.select_dtypes(
    include=np.number
).columns.drop("is_faulted")

numeric_features

In [ ]:
# Descriptive stats (mean, std, min, max) for numeric columns.
df[numeric_features].describe()

In [ ]:
# Histograms for all numeric features.
df[numeric_features].hist(
    bins=30,
    figsize=(15, 10)
)

plt.tight_layout()
plt.show()

In [ ]:
# Skewness of numeric features, sorted descending.
skewness = df[numeric_features].skew().sort_values(ascending=False)

print(skewness)

### Outliers Overview

In [ ]:
# Continuous features only (more than 2 unique values, not 0/1 flags).
continuous_features = [
    col for col in numeric_features
    if df[col].nunique() > 2
]

continuous_features

In [ ]:
# Boxplots for all continuous features to spot outliers.
df[continuous_features].plot(
    kind="box",
    subplots=True,
    layout=(4, 4),
    figsize=(18, 14)
)

plt.tight_layout()
plt.show()

### Explore Outliers

In [ ]:
# Define check_outliers(): boxplot + IQR-based outlier bounds/count for a column.
def check_outliers(data, col, symmetric=True):
    """Boxplot + IQR-based outlier summary for one numeric column.

    symmetric=True checks both tails (lower and upper bound).
    symmetric=False checks the upper tail only (for columns where only
    unusually HIGH values are meaningful outliers, e.g. irradiance).

    Returns the bounds and outlier count so results can be inspected later,
    instead of just printing them.
    """
    plt.figure(figsize=(14, 5))
    sns.boxplot(x=data[col])
    plt.title(f"Boxplot of {col}", fontsize=18)
    plt.xlabel(col, fontsize=14)
    plt.tight_layout()
    plt.show()

    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    if symmetric:
        mask = (data[col] < lower_bound) | (data[col] > upper_bound)
    else:
        mask = data[col] > upper_bound
    outliers = data[col][mask]

    print(f"--- {col} ---")
    print("Q1:", Q1, "| Q3:", Q3, "| IQR:", IQR)
    print("Lower bound:", lower_bound, "| Upper bound:", upper_bound)
    print("Number of outliers:", len(outliers))
    if len(outliers) > 0:
        print("Min outlier:", outliers.min(), "| Max outlier:", outliers.max())
    print()

    return {"lower_bound": lower_bound, "upper_bound": upper_bound, "n_outliers": len(outliers)}

In [ ]:
# Run check_outliers() on 4 features with outliers; keep them, they're valid readings.
outlier_summary = {}
outlier_summary["irradiance"] = check_outliers(df, "irradiance", symmetric=False)
outlier_summary["module_temp"] = check_outliers(df, "module_temp")
outlier_summary["inverter_temp"] = check_outliers(df, "inverter_temp")
outlier_summary["cloud_cover"] = check_outliers(df, "cloud_cover")

outlier_summary

## Correlation Analysis

In [ ]:
# Correlation heatmap between continuous features and is_faulted.
plt.figure(figsize=(16, 12))
plt.title("Correlation Heatmap", fontsize=20)
sns.heatmap(
    df[continuous_features + ["is_faulted"]].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.tight_layout()
plt.show()

## Convert Strings to Datetime & Sort

In [ ]:
# Convert time columns to datetime, sort by time.
df['downtime_start_time'] = pd.to_datetime(df['downtime_start_time'])
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values('time')

In [ ]:
# Engineer hour, month, day_of_week, is_daylight; set time as index.
df['hour'] = df['time'].dt.hour
df['month'] = df['time'].dt.month
df['day_of_week'] = df['time'].dt.dayofweek
df['is_daylight'] = df['hour'].between(6, 18).astype(int)
df = df.set_index('time')

## Time Gaps Check

In [ ]:
# Compute time gap (minutes) between consecutive readings per device.
time_series = df.index.to_series()
df["time_diff_min"] = time_series.groupby(df["device"]).diff().dt.total_seconds() / 60.0

In [ ]:
# Find the expected gap and count gaps larger than expected per device.
expected_gap_min = df["time_diff_min"].mode()[0]

gaps = df[df["time_diff_min"] > expected_gap_min]
gaps_per_device = gaps.groupby("device").size()

print("Expected gap between readings (minutes):", expected_gap_min)
print("Total gaps found:", len(gaps))
gaps_per_device

In [ ]:
# Drop the temporary time_diff_min column.
df = df.drop(columns=["time_diff_min"])

## Calculate Downtime (minutes)

In [ ]:
# Compute downtime_duration_min, fill NaNs with 0, drop downtime_start_time.
df['downtime_duration_min'] = (df.index - df['downtime_start_time']).dt.total_seconds() / 60.0
df['downtime_duration_min'] = df['downtime_duration_min'].fillna(0).clip(lower=0)

df = df.drop(columns=['downtime_start_time'])

In [ ]:
# Preview data after adding the new time-derived columns.
df.head()

## Encode Categorical Target

In [ ]:
# Value counts for fault_labels.
df['fault_labels'].value_counts()

In [ ]:
# Value counts for fault_severity.
df['fault_severity'].value_counts()

In [ ]:
# Compute dc/ac power, inverter_efficiency, and encode fault_severity as ordinal.
df['dc_power'] = (df['dc_voltage'] * df['dc_current']) 

df['ac_power'] = (df['ac_voltage'] * df['ac_current']) 

df['inverter_efficiency'] = np.where(df['dc_power'] > 0, df['ac_power'] / df['dc_power'], 0.0)

df['inverter_efficiency'] = df['inverter_efficiency'].clip(lower=0.0, upper=1.0)

df['fault_severity'] = df['fault_severity'].map({'none':0,'low':1,'medium':2,'high':3})

## Check Class Imbalance

### is_faulted

In [ ]:
# Print and plot the distribution of is_faulted.
print(df["is_faulted"].value_counts())
print()
print(df["is_faulted"].value_counts(normalize=True) * 100)

plt.figure(figsize=(7, 5))
plt.title("Distribution of is_faulted")
sns.countplot(data=df, x="is_faulted")
plt.xlabel("is_faulted")
plt.ylabel("Count")
plt.show()

### fault_labels

In [ ]:
# Print and plot the distribution of fault_labels.
print(df["fault_labels"].value_counts())
print()
print(df["fault_labels"].value_counts(normalize=True) * 100)

plt.figure(figsize=(10, 5))
plt.title("Distribution of fault_labels")
sns.countplot(data=df, x="fault_labels", order=df["fault_labels"].value_counts().index)
plt.xlabel("fault_labels")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### fault_severity

In [ ]:
# Print and plot the distribution of fault_severity.
print(df["fault_severity"].value_counts())
print()
print(df["fault_severity"].value_counts(normalize=True) * 100)

plt.figure(figsize=(7, 5))
plt.title("Distribution of fault_severity")
sns.countplot(data=df, x="fault_severity", order=df["fault_severity"].value_counts().index)
plt.xlabel("fault_severity")
plt.ylabel("Count")
plt.show()

## Train / Test Split — General Strategy

In [ ]:
# Fault ratio per month — not uniform across the year.
monthly_fault_ratio = df.groupby(df.index.month)["is_faulted"].mean() * 100

plt.figure(figsize=(10, 5))
bars = plt.bar(monthly_fault_ratio.index, monthly_fault_ratio.values, color="#4C72B0")
plt.title("Fault Ratio per Month — NOT uniform across the year")
plt.xlabel("Month")
plt.ylabel("% is_faulted = 1")
plt.xticks(range(1, 13))
for b in bars:
    plt.text(b.get_x() + b.get_width()/2, b.get_height()+0.5, f"{b.get_height():.0f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

### Step 2: Blocked Time Split per (device × week)

In [ ]:
# Chronological 80/20 split per (device, week); build train_df / test_df.
def time_block_split(group, train_frac=0.8):
    """Chronological 80/20 split within one group (device, week).
    First `train_frac` of the rows (already sorted by time) go to train,
    the last remaining rows go to test -> train always precedes test.
    """
    n = len(group)
    cut = int(np.floor(n * train_frac))
    return group.iloc[:cut], group.iloc[cut:]

df["week"] = df.index.isocalendar().week

train_parts, test_parts = [], []
for _, g in df.groupby(["device", "week"], sort=False):
    tr, te = time_block_split(g)
    train_parts.append(tr)
    test_parts.append(te)

train_df = pd.concat(train_parts).sort_index()
test_df = pd.concat(test_parts).sort_index()

df = df.drop(columns=["week"])
train_df = train_df.drop(columns=["week"])
test_df = test_df.drop(columns=["week"])

print("train_df:", train_df.shape)
print("test_df :", test_df.shape)
print(f"train is_faulted ratio: {train_df['is_faulted'].mean()*100:.2f}%")
print(f"test  is_faulted ratio: {test_df['is_faulted'].mean()*100:.2f}%")

### Step 3: Visual Confirmation the Split Is Balanced

In [ ]:
# Visual check that is_faulted / fault_severity ratios match between train and test.
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
w = 0.35

ax = axes[0]
v_train = train_df["is_faulted"].mean() * 100
v_test = test_df["is_faulted"].mean() * 100
ax.bar(0 - w/2, v_train, w, label="Train", color="#4C72B0")
ax.bar(0 + w/2, v_test, w, label="Test", color="#DD8452")
ax.set_xticks([0]); ax.set_xticklabels(["is_faulted = 1"])
ax.set_ylabel("%"); ax.set_ylim(0, 30)
ax.set_title("is_faulted balance (weekly blocked split)")
ax.text(0 - w/2, v_train + 0.3, f"{v_train:.1f}%", ha="center")
ax.text(0 + w/2, v_test + 0.3, f"{v_test:.1f}%", ha="center")
ax.legend()

order = [0, 1, 2, 3]
labels = ["none", "low", "medium", "high"]
st = train_df["fault_severity"].value_counts(normalize=True).reindex(order) * 100
se = test_df["fault_severity"].value_counts(normalize=True).reindex(order) * 100
ax2 = axes[1]
xs = np.arange(4)
ax2.bar(xs - w/2, st.values, w, label="Train", color="#4C72B0")
ax2.bar(xs + w/2, se.values, w, label="Test", color="#DD8452")
ax2.set_xticks(xs); ax2.set_xticklabels(labels)
ax2.set_ylabel("% of rows")
ax2.set_title("fault_severity balance (weekly blocked split)")
ax2.legend()

plt.tight_layout()
plt.show()

**Result:** `is_faulted` ratio is close between train/test (~20% vs ~21%); `train_df`/`test_df` are the base for all 4 models.

## Model 1: Predict `active_power` (Regression)
Dropped: identity/target cols, anything derived from active_power (incl. `irradiance`).

In [ ]:
# Model 1 leakage cols: identity/targets + anything derived from active_power (incl. irradiance).
leakage_cols_general = [
    "device",
    "is_faulted", "fault_labels", "fault_severity",
    "fault_soiling", "fault_inverter_overheat", "fault_tracker_stuck", "fault_dc_string_outage",
    "downtime_duration_min",
]

model1_extra_leakage = [
    "active_power",
    "ac_power", "dc_power",
    "ac_voltage", "ac_current", "dc_voltage", "dc_current",
    "inverter_efficiency",
    "performance_ratio",
    "irradiance",
]

model1_drop_cols = leakage_cols_general + model1_extra_leakage
model1_features = [c for c in df.columns if c not in model1_drop_cols]

print("Number of features used in Model 1:", len(model1_features))
model1_features

In [ ]:
# Build X/y train/test for Model 1 (target = active_power).
X_train_m1 = train_df[model1_features]
y_train_m1 = train_df["active_power"]

X_test_m1 = test_df[model1_features]
y_test_m1 = test_df["active_power"]

print("X_train_m1:", X_train_m1.shape)
print("X_test_m1 :", X_test_m1.shape)
print("y_train_m1:", y_train_m1.shape)
print("y_test_m1 :", y_test_m1.shape)

X_train_m1.head()

In [ ]:
# Sanity check: correlation of remaining features with active_power (train set).
corr_with_target = X_train_m1.select_dtypes(include=np.number).corrwith(y_train_m1).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
corr_with_target.plot(kind="barh", color="#55A868")
plt.title("Correlation of remaining features with active_power (train set)")
plt.xlabel("Correlation")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

corr_with_target

## Model 2: `is_faulted` (Binary Classification)
Only identity/target leakage cols dropped — physical readings are legitimate signal here.

In [ ]:
# Model 2 features: only drop identity/target leakage cols; physical readings are legit.
model2_drop_cols = leakage_cols_general
model2_features = [c for c in df.columns if c not in model2_drop_cols]

print("Number of features used in Model 2:", len(model2_features))
model2_features

In [ ]:
# Build X/y train/test for Model 2 (target = is_faulted).
X_train_m2 = train_df[model2_features]
y_train_m2 = train_df["is_faulted"]

X_test_m2 = test_df[model2_features]
y_test_m2 = test_df["is_faulted"]

print("X_train_m2:", X_train_m2.shape)
print("X_test_m2 :", X_test_m2.shape)
print("y_train_m2:", y_train_m2.shape)
print("y_test_m2 :", y_test_m2.shape)

X_train_m2.head()

### Visual Check: Do Features Separate the Two Classes?

In [ ]:
# Boxplot: performance_ratio by fault status, confirming real signal.
plt.figure(figsize=(8, 5))
sns.boxplot(data=train_df, x="is_faulted", y="performance_ratio")
plt.title("Performance Ratio by Fault Status (train set)")
plt.xlabel("is_faulted")
plt.ylabel("performance_ratio")
plt.show()

## Model 3: `fault_labels` (Multi-class Classification)
Same drop list as Model 2; rare class `dc_string_outage|downtime` excluded from target.

In [ ]:
# Model 3 features: same as Model 2, device especially important to drop.
model3_drop_cols = leakage_cols_general
model3_features = [c for c in df.columns if c not in model3_drop_cols]

print("Number of features used in Model 3:", len(model3_features))
print("Same features as Model 2?", model3_features == model2_features)

In [ ]:
# Build X/y train/test for Model 3; exclude rare class dc_string_outage|downtime.
X_train_m3 = train_df[model3_features]
y_train_m3 = train_df["fault_labels"]

X_test_m3 = test_df[model3_features]
y_test_m3 = test_df["fault_labels"]

excluded_label_m3 = "dc_string_outage|downtime"
train_mask_m3 = y_train_m3 != excluded_label_m3
test_mask_m3 = y_test_m3 != excluded_label_m3

X_train_m3 = X_train_m3[train_mask_m3]
y_train_m3 = y_train_m3[train_mask_m3]
X_test_m3 = X_test_m3[test_mask_m3]
y_test_m3 = y_test_m3[test_mask_m3]

print("X_train_m3:", X_train_m3.shape)
print("X_test_m3 :", X_test_m3.shape)
print("y_train_m3:", y_train_m3.shape)
print("y_test_m3 :", y_test_m3.shape)

X_train_m3.head()

### Visual Check: fault_labels Balance (Train vs Test)

In [ ]:
# Compare fault_labels distribution between train/test after excluding the rare class.
train_counts_lbl = y_train_m3.value_counts(normalize=True) * 100
test_counts_lbl = y_test_m3.value_counts(normalize=True) * 100

compare_labels_df = pd.DataFrame({"Train %": train_counts_lbl, "Test %": test_counts_lbl}).fillna(0)
compare_labels_df = compare_labels_df.sort_values("Train %", ascending=False)

compare_labels_df.plot(kind="bar", figsize=(10, 5), color=["#4C72B0", "#DD8452"])
plt.title("Fault Labels Distribution — Train vs Test")
plt.xlabel("fault_labels")
plt.ylabel("% of rows")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

compare_labels_df

## Model 4: `fault_severity` (Ordinal / Multi-class Classification)
Same features as Models 2 & 3; target already ordinal-encoded.

In [ ]:
# Model 4 features: same set as Models 2 and 3, only the target differs.
model4_drop_cols = leakage_cols_general
model4_features = [c for c in df.columns if c not in model4_drop_cols]

print("Number of features used in Model 4:", len(model4_features))
print("Same features as Models 2 and 3?", model4_features == model2_features == model3_features)

In [ ]:
# Build X/y train/test for Model 4 (target = fault_severity).
X_train_m4 = train_df[model4_features]
y_train_m4 = train_df["fault_severity"]

X_test_m4 = test_df[model4_features]
y_test_m4 = test_df["fault_severity"]

print("X_train_m4:", X_train_m4.shape)
print("X_test_m4 :", X_test_m4.shape)
print("y_train_m4:", y_train_m4.shape)
print("y_test_m4 :", y_test_m4.shape)

X_train_m4.head()

### Visual Check: fault_severity Balance (Train vs Test)

In [ ]:
# Compare fault_severity distribution between train and test.
sev_labels = {0: "none", 1: "low", 2: "medium", 3: "high"}

train_counts_sev = y_train_m4.value_counts(normalize=True).reindex([0, 1, 2, 3]) * 100
test_counts_sev = y_test_m4.value_counts(normalize=True).reindex([0, 1, 2, 3]) * 100

compare_sev_df = pd.DataFrame({"Train %": train_counts_sev, "Test %": test_counts_sev})
compare_sev_df.index = compare_sev_df.index.map(sev_labels)

compare_sev_df.plot(kind="bar", figsize=(8, 5), color=["#4C72B0", "#DD8452"])
plt.title("Fault Severity Distribution — Train vs Test")
plt.xlabel("fault_severity")
plt.ylabel("% of rows")
plt.tight_layout()
plt.show()

compare_sev_df

## Summary: Ready for Modeling
4 datasets built on the same balanced time split, differing only by target and allowed features.

# Part 2: Modeling
CatBoost & LightGBM, model by model: train, evaluate, plot, interpret.

In [ ]:
# Import CatBoost, LightGBM, and evaluation metrics.
from catboost import CatBoostRegressor, CatBoostClassifier
from lightgbm import LGBMRegressor, LGBMClassifier

from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

## Model 1: Train & Evaluate — CatBoost vs LightGBM (Regression)

### Evaluation Metrics: MAE, RMSE, R2
Lower MAE/RMSE is better; R2 closer to 1 is better.

In [ ]:
# Train CatBoost Regressor for Model 1, predict on test set.

cat_reg_m1 = CatBoostRegressor(
    iterations=600,
    learning_rate=0.05,
    depth=8,
    loss_function="RMSE",
    random_state=42,
    verbose=False
)

cat_reg_m1.fit(X_train_m1, y_train_m1)

pred_cat_m1 = cat_reg_m1.predict(X_test_m1)

In [ ]:
# Train LightGBM Regressor for Model 1, predict on test set.

lgbm_reg_m1 = LGBMRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    random_state=42,
    verbose=-1
)

lgbm_reg_m1.fit(X_train_m1, y_train_m1)

pred_lgbm_m1 = lgbm_reg_m1.predict(X_test_m1)

In [ ]:
# Define regression_report(): MAE/RMSE/R2 helper; evaluate both models for Model 1.

def regression_report(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"--- {model_name} (Model 1: active_power) ---")
    print(f"MAE  : {mae:.3f}")
    print(f"RMSE : {rmse:.3f}")
    print(f"R2   : {r2:.4f}")
    print()

    return {"Model": model_name, "MAE": mae, "RMSE": rmse, "R2": r2}

results_m1 = []
results_m1.append(regression_report(y_test_m1, pred_cat_m1, "CatBoost"))
results_m1.append(regression_report(y_test_m1, pred_lgbm_m1, "LightGBM"))

results_m1_df = pd.DataFrame(results_m1)
results_m1_df

### Chart 1: Compare the 3 Metrics Side by Side

In [ ]:
# Bar chart comparing MAE, RMSE, R2 between CatBoost and LightGBM (Model 1).
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
metrics_m1 = ["MAE", "RMSE", "R2"]
colors_m1 = ["#4C72B0", "#DD8452"]

for ax, metric in zip(axes, metrics_m1):
    bars = ax.bar(results_m1_df["Model"], results_m1_df[metric], color=colors_m1)
    ax.set_title(metric, fontsize=14)
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height(), f"{b.get_height():.3f}",
                 ha="center", va="bottom", fontsize=10)

plt.suptitle("Model 1 (active_power) — CatBoost vs LightGBM", fontsize=15)
plt.tight_layout()
plt.show()

### Chart 2: Predicted vs Actual
Closer to the red y=x line means more accurate predictions.

In [ ]:
# Scatter plot: predicted vs actual active_power, with a y=x reference line.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

preds_m1 = {"CatBoost": pred_cat_m1, "LightGBM": pred_lgbm_m1}
max_val = max(y_test_m1.max(), pred_cat_m1.max(), pred_lgbm_m1.max())

for ax, (name, pred) in zip(axes, preds_m1.items()):
    ax.scatter(y_test_m1, pred, alpha=0.08, s=6, color="#4C72B0")
    ax.plot([0, max_val], [0, max_val], color="red", linestyle="--", linewidth=1.5, label="Perfect prediction (y = x)")
    ax.set_xlabel("Actual active_power")
    ax.set_ylabel("Predicted active_power")
    ax.set_title(name)
    ax.legend()

plt.suptitle("Model 1 — Predicted vs Actual (test set)", fontsize=15)
plt.tight_layout()
plt.show()

### Chart 3: Residuals Distribution
Centered near zero and narrow means low, unbiased error.

In [ ]:
# Residuals (actual - predicted) distribution for both models.
residuals_cat_m1 = y_test_m1 - pred_cat_m1
residuals_lgbm_m1 = y_test_m1 - pred_lgbm_m1

plt.figure(figsize=(10, 5))
sns.histplot(residuals_cat_m1, color="#4C72B0", label="CatBoost", kde=True, stat="density", alpha=0.4, bins=60)
sns.histplot(residuals_lgbm_m1, color="#DD8452", label="LightGBM", kde=True, stat="density", alpha=0.4, bins=60)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.title("Model 1 — Residuals distribution (Actual − Predicted)")
plt.xlabel("Residual")
plt.legend()
plt.tight_layout()
plt.show()

### Chart 4: Feature Importance
Sanity check against leakage — physical/time features should dominate, not `irradiance` (it's excluded from Model 1 now).

In [ ]:
# Feature importance plots for CatBoost and LightGBM (Model 1).
cat_importance_m1 = pd.Series(
    cat_reg_m1.get_feature_importance(), index=X_train_m1.columns
).sort_values(ascending=False)

lgbm_importance_m1 = pd.Series(
    lgbm_reg_m1.feature_importances_, index=X_train_m1.columns
).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

cat_importance_m1.plot(kind="barh", ax=axes[0], color="#55A868")
axes[0].set_title("CatBoost — Feature Importance (Model 1)")
axes[0].invert_yaxis()

lgbm_importance_m1.plot(kind="barh", ax=axes[1], color="#C44E52")
axes[1].set_title("LightGBM — Feature Importance (Model 1)")
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Model 1 Summary
Compare MAE/RMSE/R2, Predicted-vs-Actual fit, residuals, and feature importance to judge the winner.

## Model 2

In [ ]:
# Summary: sample counts, feature count, target distribution for Model 2.

print("=" * 70)
print("MODEL 2 : Fault Detection")
print("=" * 70)

print()

print("Training samples :", X_train_m2.shape[0])
print("Testing samples  :", X_test_m2.shape[0])

print()

print("Number of Features :", X_train_m2.shape[1])

print()

print("Target Distribution (Train)")
print(y_train_m2.value_counts())

print()

print("Target Distribution (%)")
print(round(y_train_m2.value_counts(normalize=True) * 100,2))

In [ ]:
# Table listing feature names used in Model 2.
features_df = pd.DataFrame({

    "Feature":X_train_m2.columns

})

features_df

In [ ]:
# Compute balanced class weights for is_faulted imbalance.
from sklearn.utils.class_weight import compute_class_weight

classes_m2 = np.unique(y_train_m2)

weights_m2 = compute_class_weight(

    class_weight="balanced",

    classes=classes_m2,

    y=y_train_m2

)

class_weights_m2 = dict(zip(classes_m2, weights_m2))

print(class_weights_m2)

In [ ]:
# Bar chart of class weights (Healthy / Fault) for Model 2.
plt.figure(figsize=(6,4))

plt.bar(

    ["Healthy","Fault"],

    class_weights_m2.values(),

    color=["steelblue","tomato"]

)

plt.title("Class Weights")

plt.ylabel("Weight")

plt.show()

In [ ]:
# Train CatBoost Classifier on Model 2 data with class weights.

cat_model_m2 = CatBoostClassifier(

    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="F1",
    class_weights=class_weights_m2,
    random_state=42,
    verbose=100

)

cat_model_m2.fit(

    X_train_m2,
    y_train_m2

)

In [ ]:
# Train LightGBM Classifier on Model 2 data with class weights.

lgbm_model_m2 = LGBMClassifier(

    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    class_weight=class_weights_m2,
    random_state=42

)

lgbm_model_m2.fit(

    X_train_m2,
    y_train_m2

)

In [ ]:
# Predict on the test set with both models for Model 2.
pred_cat_m2 = cat_model_m2.predict(X_test_m2)

pred_lgbm_m2 = lgbm_model_m2.predict(X_test_m2)

In [ ]:
# Import classification metrics (accuracy, f1, report, confusion matrix).

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
# Print classification reports for both models (Model 2).

print("=" * 80)
print("CatBoost Classification Report")
print("=" * 80)

print(classification_report(
    y_test_m2,
    pred_cat_m2,
    target_names=["Healthy", "Fault"],
    digits=4
))

print("\n")

print("=" * 80)
print("LightGBM Classification Report")
print("=" * 80)

print(classification_report(
    y_test_m2,
    pred_lgbm_m2,
    target_names=["Healthy", "Fault"],
    digits=4
))

In [ ]:
# Compute confusion matrices for both models (Model 2).

cm_cat = confusion_matrix(
    y_test_m2,
    pred_cat_m2
)

cm_lgb = confusion_matrix(
    y_test_m2,
    pred_lgbm_m2
)

In [ ]:
# Plot both confusion matrices as heatmaps side by side.
fig,axes = plt.subplots(
    1,
    2,
    figsize=(13,5)
)

sns.heatmap(
    cm_cat,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Healthy","Fault"],
    yticklabels=["Healthy","Fault"],
    ax=axes[0]
)

axes[0].set_title("CatBoost")

axes[0].set_xlabel("Predicted")

axes[0].set_ylabel("Actual")

sns.heatmap(
    cm_lgb,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["Healthy","Fault"],
    yticklabels=["Healthy","Fault"],
    ax=axes[1]
)

axes[1].set_title("LightGBM")

axes[1].set_xlabel("Predicted")

axes[1].set_ylabel("Actual")

plt.tight_layout()

plt.show()

In [ ]:
# Top-10 feature importance table from CatBoost (Model 2).
importance_cat = pd.DataFrame({

    "Feature":X_train_m2.columns,

    "Importance":cat_model_m2.get_feature_importance()

})

importance_cat = importance_cat.sort_values(
    by="Importance",
    ascending=False
)

importance_cat.head(10)

In [ ]:
# Bar chart of top-10 CatBoost feature importances (Model 2).
plt.figure(figsize=(10,6))

sns.barplot(

    data=importance_cat.head(10),

    x="Importance",

    y="Feature"

)

plt.title("Top 10 Important Features - CatBoost")

plt.show()

In [ ]:
# Top-10 feature importance table from LightGBM (Model 2).
importance_lgb = pd.DataFrame({

    "Feature":X_train_m2.columns,

    "Importance":lgbm_model_m2.feature_importances_

})

importance_lgb = importance_lgb.sort_values(
    by="Importance",
    ascending=False
)

importance_lgb.head(10)

In [ ]:
# Bar chart of top-10 LightGBM feature importances (Model 2).
plt.figure(figsize=(10,6))

sns.barplot(

    data=importance_lgb.head(10),

    x="Importance",

    y="Feature"

)

plt.title("Top 10 Important Features - LightGBM")

plt.show()

## Model 3

In [ ]:
# Rebuild Model 3 data before training; exclude the rare combined class again.
model3_drop_cols = leakage_cols_general
model3_features = [c for c in df.columns if c not in model3_drop_cols]

X_train_m3 = train_df[model3_features]
y_train_m3 = train_df["fault_labels"]
X_test_m3 = test_df[model3_features]
y_test_m3 = test_df["fault_labels"]

excluded_label_m3 = "dc_string_outage|downtime"
train_mask_m3 = y_train_m3 != excluded_label_m3
test_mask_m3 = y_test_m3 != excluded_label_m3

X_train_m3 = X_train_m3[train_mask_m3]
y_train_m3 = y_train_m3[train_mask_m3]
X_test_m3 = X_test_m3[test_mask_m3]
y_test_m3 = y_test_m3[test_mask_m3]

print("Training samples :", X_train_m3.shape[0])
print("Testing samples  :", X_test_m3.shape[0])
print("Number of Features :", X_train_m3.shape[1])
print("\nTarget Distribution in Train Set:")
print(y_train_m3.value_counts())
print("\nTarget Distribution (%) in Train Set:")
print(round(y_train_m3.value_counts(normalize=True) * 100, 3))

In [ ]:
# Compute balanced class weights for fault-type imbalance (Model 3).
classes_m3 = np.unique(y_train_m3)
weights_m3 = compute_class_weight(
    class_weight="balanced", 
    classes=classes_m3, 
    y=y_train_m3
)
class_weights_m3 = dict(zip(classes_m3, weights_m3))

print("\nComputed Class Weights:")
for cls, w in class_weights_m3.items():
    print(f"{cls:25s}: {w:.4f}")

plt.figure(figsize=(10, 4))
plt.bar(class_weights_m3.keys(), class_weights_m3.values(), color="teal")
plt.title("Calculated Class Weights for Multi-class Imbalance (Model 3)")
plt.ylabel("Weight")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Train multi-class CatBoost Classifier on Model 3 data.
cat_model_m3 = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    class_weights=class_weights_m3,
    random_state=42,
    verbose=100
)

print("\nTraining CatBoost Model 3...")
cat_model_m3.fit(X_train_m3, y_train_m3)

In [ ]:
# Train multi-class LightGBM Classifier on Model 3 data.
lgbm_model_m3 = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    objective="multiclass",
    class_weight=class_weights_m3,
    random_state=42
)

print("\nTraining LightGBM Model 3...")
lgbm_model_m3.fit(X_train_m3, y_train_m3)

In [ ]:
# Predict on the test set with both models for Model 3.
pred_cat_m3 = cat_model_m3.predict(X_test_m3).ravel()
pred_lgb_m3 = lgbm_model_m3.predict(X_test_m3)

In [ ]:
# Print classification reports for both models (Model 3).
print("=" * 80)
print("CatBoost Model 3 - Classification Report")
print("=" * 80)
print(classification_report(y_test_m3, pred_cat_m3, digits=4))

print("=" * 80)
print("LightGBM Model 3 - Classification Report")
print("=" * 80)
print(classification_report(y_test_m3, pred_lgb_m3, digits=4))

In [ ]:
# Confusion matrices heatmaps for both models (Model 3).
cm_cat_m3 = confusion_matrix(y_test_m3, pred_cat_m3, labels=classes_m3)
cm_lgb_m3 = confusion_matrix(y_test_m3, pred_lgb_m3, labels=classes_m3)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(
    cm_cat_m3, annot=True, fmt="d", cmap="Blues",
    xticklabels=classes_m3, yticklabels=classes_m3, ax=axes[0]
)
axes[0].set_title("CatBoost - Fault Labels Confusion Matrix")
axes[0].set_xlabel("Predicted Label")
axes[0].set_ylabel("Actual Label")
axes[0].tick_params(axis='x', rotation=45)

sns.heatmap(
    cm_lgb_m3, annot=True, fmt="d", cmap="Greens",
    xticklabels=classes_m3, yticklabels=classes_m3, ax=axes[1]
)
axes[1].set_title("LightGBM - Fault Labels Confusion Matrix")
axes[1].set_xlabel("Predicted Label")
axes[1].set_ylabel("Actual Label")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Top-10 feature importance bar charts for both models (Model 3).
importance_cat_m3 = pd.DataFrame({
    "Feature": X_train_m3.columns,
    "Importance": cat_model_m3.get_feature_importance()
}).sort_values(by="Importance", ascending=False)

importance_lgb_m3 = pd.DataFrame({
    "Feature": X_train_m3.columns,
    "Importance": lgbm_model_m3.feature_importances_
}).sort_values(by="Importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=importance_cat_m3.head(10), x="Importance", y="Feature", ax=axes[0], color="steelblue")
axes[0].set_title("Top 10 Important Features - CatBoost (Model 3)")

sns.barplot(data=importance_lgb_m3.head(10), x="Importance", y="Feature", ax=axes[1], color="seagreen")
axes[1].set_title("Top 10 Important Features - LightGBM (Model 3)")

plt.tight_layout()
plt.show()

## Model 4

In [ ]:
# Summary: sample counts, feature count, target distribution for Model 4.
print("=" * 70)
print("MODEL 4 : Fault Severity Detection")
print("=" * 70)
print()
print("Training samples :", X_train_m4.shape[0])
print("Testing samples :", X_test_m4.shape[0])
print("Number of Features :", X_train_m4.shape[1])
print("\nTarget Distribution in Train Set:")
print(y_train_m4.value_counts())
print("\nTarget Distribution (%) in Train Set:")
print(round(y_train_m4.value_counts(normalize=True) * 100, 3))

In [ ]:
# Compute balanced class weights for severity imbalance (Model 4).
classes_m4 = np.unique(y_train_m4)
weights_m4 = compute_class_weight(
    class_weight="balanced",
    classes=classes_m4,
    y=y_train_m4
)
class_weights_m4 = dict(zip(classes_m4, weights_m4))

print("\nComputed Class Weights for Model 4:")
for cls, w in class_weights_m4.items():
    print(f"Class {cls}: {w:.4f}")

plt.figure(figsize=(8, 4))
plt.bar([str(c) for c in class_weights_m4.keys()], class_weights_m4.values(), color="purple")
plt.title("Calculated Class Weights for Multi-class Imbalance (Model 4)")
plt.xlabel("Fault Severity Class (0: None, 1: Low, 2: Medium, 3: High)")
plt.ylabel("Weight")
plt.tight_layout()
plt.show()

In [ ]:
# Train multi-class CatBoost Classifier on Model 4 data.
cat_model_m4 = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    class_weights=class_weights_m4,
    random_state=42,
    verbose=100
)

print("\nTraining CatBoost Model 4...")
cat_model_m4.fit(X_train_m4, y_train_m4)

In [ ]:
# Train multi-class LightGBM Classifier on Model 4 data.
lgbm_model_m4 = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    objective="multiclass",
    class_weight=class_weights_m4,
    random_state=42
)

print("\nTraining LightGBM Model 4...")
lgbm_model_m4.fit(X_train_m4, y_train_m4)

In [ ]:
# Predict on test set and print classification reports (Model 4).
pred_cat_m4 = cat_model_m4.predict(X_test_m4).ravel()
pred_lgb_m4 = lgbm_model_m4.predict(X_test_m4)

print("\n" + "=" * 80)
print("CatBoost Model 4 - Classification Report")
print("=" * 80)
print(classification_report(y_test_m4, pred_cat_m4, digits=4))

print("=" * 80)
print("LightGBM Model 4 - Classification Report")
print("=" * 80)
print(classification_report(y_test_m4, pred_lgb_m4, digits=4))

In [ ]:
# Confusion matrices heatmaps for both models (Model 4).
cm_cat_m4 = confusion_matrix(y_test_m4, pred_cat_m4, labels=classes_m4)
cm_lgb_m4 = confusion_matrix(y_test_m4, pred_lgb_m4, labels=classes_m4)

severity_labels = ["None (0)", "Low (1)", "Medium (2)", "High (3)"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(
    cm_cat_m4, annot=True, fmt="d", cmap="Blues",
    xticklabels=severity_labels, yticklabels=severity_labels, ax=axes[0]
)
axes[0].set_title("CatBoost - Fault Severity Confusion Matrix")
axes[0].set_xlabel("Predicted Severity")
axes[0].set_ylabel("Actual Severity")

sns.heatmap(
    cm_lgb_m4, annot=True, fmt="d", cmap="Greens",
    xticklabels=severity_labels, yticklabels=severity_labels, ax=axes[1]
)
axes[1].set_title("LightGBM - Fault Severity Confusion Matrix")
axes[1].set_xlabel("Predicted Severity")
axes[1].set_ylabel("Actual Severity")

plt.tight_layout()
plt.show()

In [ ]:
# Top-10 feature importance bar charts for both models (Model 4).
importance_cat_m4 = pd.DataFrame({
    "Feature": X_train_m4.columns,
    "Importance": cat_model_m4.get_feature_importance()
}).sort_values(by="Importance", ascending=False)

importance_lgb_m4 = pd.DataFrame({
    "Feature": X_train_m4.columns,
    "Importance": lgbm_model_m4.feature_importances_
}).sort_values(by="Importance", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=importance_cat_m4.head(10), x="Importance", y="Feature", ax=axes[0], color="steelblue")
axes[0].set_title("Top 10 Important Features - CatBoost (Model 4)")

sns.barplot(data=importance_lgb_m4.head(10), x="Importance", y="Feature", ax=axes[1], color="mediumseagreen")
axes[1].set_title("Top 10 Important Features - LightGBM (Model 4)")

plt.tight_layout()
plt.show()

## Evaluate All Models

In [ ]:
# Compare classification models 2/3/4 only (Model 1 excluded — regression bug fix).
models_info = [
    ("Model 2: Fault Cause", y_test_m2, pred_cat_m2, pred_lgbm_m2),
    ("Model 3: Component at Fault", y_test_m3, pred_cat_m3, pred_lgb_m3),
    ("Model 4: Fault Severity", y_test_m4, pred_cat_m4, pred_lgb_m4)
]

evaluation_results = []

for model_name, y_true, pred_cat, pred_lgb in models_info:

    acc_cat = accuracy_score(y_true, pred_cat)
    f1_macro_cat = f1_score(y_true, pred_cat, average='macro')
    f1_weighted_cat = f1_score(y_true, pred_cat, average='weighted')

    evaluation_results.append({
        "Model Task": model_name,
        "Algorithm": "CatBoost",
        "Accuracy": acc_cat,
        "F1-Score (Macro)": f1_macro_cat,
        "F1-Score (Weighted)": f1_weighted_cat
    })

    acc_lgb = accuracy_score(y_true, pred_lgb)
    f1_macro_lgb = f1_score(y_true, pred_lgb, average='macro')
    f1_weighted_lgb = f1_score(y_true, pred_lgb, average='weighted')

    evaluation_results.append({
        "Model Task": model_name,
        "Algorithm": "LightGBM",
        "Accuracy": acc_lgb,
        "F1-Score (Macro)": f1_macro_lgb,
        "F1-Score (Weighted)": f1_weighted_lgb
    })

results_df = pd.DataFrame(evaluation_results)

print("=" * 85)
print("MODEL 1 (Regression - active_power) IS EVALUATED SEPARATELY WITH MAE / RMSE / R2")
print("=" * 85)
display(results_m1_df)

print("=" * 85)
print("COMPREHENSIVE MODELS EVALUATION SUMMARY (Classification Models: 2, 3, 4)")
print("=" * 85)
display(results_df.style.format({
    "Accuracy": "{:.4f}",
    "F1-Score (Macro)": "{:.4f}",
    "F1-Score (Weighted)": "{:.4f}"
}).background_gradient(cmap='viridis', subset=['F1-Score (Macro)', 'Accuracy']))

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(2, 1, figsize=(14, 12))

sns.barplot(
    data=results_df,
    x="Model Task",
    y="F1-Score (Macro)",
    hue="Algorithm",
    palette=["#1f77b4", "#2ca02c"],
    ax=axes[0]
)
axes[0].set_title("Comparison of F1-Score (Macro) Across Classification Models", fontsize=16, fontweight='bold')
axes[0].set_ylabel("F1-Score (Macro)", fontsize=12)
axes[0].set_xlabel("")
axes[0].set_ylim(0, 1.1)
axes[0].legend(title="Algorithm", loc='upper right')

for p in axes[0].patches:
    axes[0].annotate(format(p.get_height(), '.3f'),
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha = 'center', va = 'center',
                     xytext = (0, 9),
                     textcoords = 'offset points',
                     fontsize=10)

sns.barplot(
    data=results_df,
    x="Model Task",
    y="Accuracy",
    hue="Algorithm",
    palette=["#1f77b4", "#2ca02c"],
    ax=axes[1]
)
axes[1].set_title("Comparison of Accuracy Across Classification Models", fontsize=16, fontweight='bold')
axes[1].set_ylabel("Accuracy", fontsize=12)
axes[1].set_xlabel("Model Task", fontsize=12)
axes[1].set_ylim(0, 1.1)
axes[1].legend(title="Algorithm", loc='upper right')

for p in axes[1].patches:
    axes[1].annotate(format(p.get_height(), '.3f'),
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha = 'center', va = 'center',
                     xytext = (0, 9),
                     textcoords = 'offset points',
                     fontsize=10)

plt.tight_layout()
plt.show()